# Notebook 05 — Diagnose validation errors before the next training experiment

The first adapter learned better answer formatting, but word-problem accuracy did not consistently improve. This notebook investigates that result using **50 fixed original validation questions**, the original base model, and the selected first adapter.

Each model answers each question with zero-shot and three-shot prompts: **200 predictions total**. No new training or test-benchmark evaluation runs. The official v2 stopping/scoring rules remain fixed.

**Kaggle:** enable Internet, select a T4, and attach only:

1. **Original Notebook 01 saved output** — data bundle.
2. **Notebook 03 saved output** — full-run selected adapter and manifests.

Notebook 02c and 04 outputs are not required. Questions were excluded from gradient training, but the full validation split was used for validation-loss checkpoint selection. These results are development diagnosis, not an untouched final holdout.


## 1. Fetch shared code

The exact GitHub commit is recorded. Keep the original data bundle and first adapter unchanged.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

REPOSITORY = "https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git"
PROJECT_REF = os.environ.get("MATH_PROJECT_REF", "main")
if PROJECT_REF.startswith("-"):
    raise ValueError("Use a branch, tag, or full commit SHA as MATH_PROJECT_REF")
PROJECT = Path(tempfile.mkdtemp(prefix="math-project-"))
subprocess.run(["git", "init", "--quiet", str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "remote", "add", "origin", REPOSITORY], check=True)
subprocess.run(["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", PROJECT_REF], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "--quiet", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPOSITORY)
print("Code commit:", CODE_COMMIT)
print("Project checkout:", PROJECT)

## 2. Find inputs and verify the diagnostic scope

The sample is selected by SHA-256 ranking of seed and original validation ID, independently of answers or model correctness. The same 50 questions are used in both modes and for both models.

Three-shot uses the original frozen training demonstrations. Zero-shot provides instructions and the problem only. Reference validation solutions are displayed later for review and are never supplied to the generator.


In [ ]:
import hashlib
import shutil

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings.")
_ = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=True)
BUNDLE = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_bundle.py"), os.environ.get("MATH_BUNDLE_PATH", "/kaggle/input")], text=True).strip())
TRAINING = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_results.py"), os.environ.get("MATH_TRAINING_PATH", "/kaggle/input"), "--kind", "training"], text=True).strip())
CONFIG_PATH = PROJECT / "configs/validation_analysis.json"
CONFIG_HASH = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
verified = subprocess.check_output([sys.executable, str(PROJECT / "scripts/analyze_validation.py"),
    "--bundle", str(BUNDLE), "--training", str(TRAINING), "--config", str(CONFIG_PATH),
    "--output", "/kaggle/working/validation-verification-only", "--stage", "verify"], text=True, cwd=PROJECT)
VERIFIED = json.loads(verified)
OUTPUT = Path("/kaggle/working") / f"validation-analysis-{VERIFIED['bundle_sha256'][:8]}-{VERIFIED['training_sha256'][:8]}-{CONFIG_HASH[:8]}-{CODE_COMMIT[:8]}"
print("Data:", BUNDLE)
print("Training:", TRAINING)
print(json.dumps(VERIFIED, indent=2))
print("Output:", OUTPUT)


## 3. Prepare the known inference/adapter environment

Use one T4 with the pinned PyTorch, Transformers, and PEFT versions. Both models run sequentially in separate processes to release GPU memory between them. Generation is greedy, batch four, with the reviewed 512-token cap and per-row stopping.

The existing nonfatal `wrapt` startup and `torch_dtype` deprecation warnings may appear. CUDA execution and CPU contract tests must still pass.


In [ ]:
ENVIRONMENT = Path(tempfile.mkdtemp(prefix="math-validation-env-"))
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/setup_environment.py"),
                    "--environment", str(ENVIRONMENT), "--profile", "training"], check=True)
PYTHON = ENVIRONMENT / "bin" / "python"
INFERENCE_ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
probe = """
import torch, peft
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable.')
x = torch.ones((32, 32), device='cuda', dtype=torch.float16)
assert (x @ x).sum().item() == 32768
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), peft.__version__)
"""
_ = subprocess.run([str(PYTHON), "-I", "-c", probe], check=True, env=INFERENCE_ENV)


## 4. Run CPU contract tests

Tiny local models/adapters check validation-only scope, deterministic selection, literal-equality flags, identity protection, interruption recovery, and both prompt modes. These tests do not estimate SmolLM2 accuracy.


In [ ]:
_ = subprocess.run([str(PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", "test_validation_analysis.py", "-b", "-v"],
                    cwd=PROJECT, check=True, env=INFERENCE_ENV)


## 5. Generate original-model answers — 100 predictions

Generate 50 zero-shot and 50 three-shot answers. Save each batch so an interrupted stage can resume in the same environment. Changed inputs, code, or runtime are refused. Original data and adapter inputs remain read-only.


In [ ]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/analyze_validation.py"),
    "--bundle", str(BUNDLE), "--training", str(TRAINING), "--config", str(CONFIG_PATH),
    "--output", str(OUTPUT), "--stage", "base"], check=True, cwd=PROJECT, env=INFERENCE_ENV)


## 6. Generate selected-adapter answers — 100 predictions

Use the same questions, prompts, stopping, and scoring. The original model's core runtime must match. This stage loads the full selected first adapter without changing its weights.


In [ ]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/analyze_validation.py"),
    "--bundle", str(BUNDLE), "--training", str(TRAINING), "--config", str(CONFIG_PATH),
    "--output", str(OUTPUT), "--stage", "tuned"], check=True, cwd=PROJECT, env=INFERENCE_ENV)


## 7. Review failures and save the diagnostic

The report separates accuracy, parseability, cap hits, repetition, and literal numeric-equality flags for all four model/mode combinations. Expand paired answers beside the reference worked solution. Inspect improvements and regressions.

The arithmetic checker recognizes only a whole line such as `52 + 18 = 60`. It never executes arbitrary expressions or repairs the answer. A mismatch is a review flag: it may involve rounding, a quoted calculation, or an actual arithmetic error. Narrative calculations, variables, and more complex expressions remain unchecked. Correct final answers can still have flawed reasoning.

Use a **copy** of `review_template.csv` to annotate wrong operations/quantities, arithmetic errors, format failures, repetitive nonanswers, reasoning concerns, or ambiguous cases. Automatic flags do not assign reasoning-error causes.

This notebook ends with review evidence. Choose the next controlled training change after inspecting validation failures. Keep the original test results fixed; do not tune to the 300 inspected pilot questions.


In [ ]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/analyze_validation.py"),
    "--bundle", str(BUNDLE), "--training", str(TRAINING), "--config", str(CONFIG_PATH),
    "--output", str(OUTPUT), "--stage", "report"], check=True, cwd=PROJECT, env=INFERENCE_ENV)
SUMMARY = json.loads((OUTPUT / "summary.json").read_text())
RESULT = json.loads((OUTPUT / "result_manifest.json").read_text())
for name, digest in RESULT["files_sha256"].items():
    if hashlib.sha256((OUTPUT / name).read_bytes()).hexdigest() != digest:
        raise RuntimeError(f"Validation artifact checksum mismatch: {name}")
from IPython.display import HTML, display
display(HTML((OUTPUT / "report.html").read_text()))
print(json.dumps(SUMMARY, indent=2))
ARCHIVE = Path(shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT))
print("Diagnostic ZIP:", ARCHIVE)
print("Save Version with outputs. Share summary.json and paired_predictions.jsonl, or the executed notebook.")
